# 🧠 GoogLeNet(InceptionV3) & ResNet50 실습 (Keras Applications)

이 노트북에서는 Keras Applications을 활용하여 GoogLeNet(InceptionV3)과 ResNet50을 실습합니다.

## 주요 내용
1. 사전학습(Pretrained) 모델 불러오기
2. 모델 튜닝 (Fine-tuning)
3. 전이학습 (Transfer Learning)
4. 파라미터 수 비교

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3, ResNet50
from tensorflow.keras.applications.inception_v3 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
from tensorflow.keras import layers, models, optimizers
import numpy as np
print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.21.0


## [1] InceptionV3 모델 불러오기

In [2]:
inception = InceptionV3(weights='imagenet', include_top=True)
inception.summary()

Model: "inception_v3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 149, 149,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 149, 149,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 149, 149,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 147, 147,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 147, 147,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 73, 73,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 73, 73,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 73, 73,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 73, 73,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 71, 71,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 71, 71,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 71, 71,    │          0 │ batch_normalizat

 Total params: 23,851,784 (90.99 MB)

 Trainable params: 23,817,352 (90.86 MB)

 Non-trainable params: 34,432 (134.50 KB)

### 예시 이미지로 예측

In [3]:
img_path = tf.keras.utils.get_file('elephant.jpg', 'https://storage.googleapis.com/download.tensorflow.org/example_images/elephant.jpg')
img = image.load_img(img_path, target_size=(299, 299))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)
preds = inception.predict(x)
print('예측 결과:', decode_predictions(preds, top=3)[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
예측 결과: [('n02504458', 'African_elephant', np.float32(0.7123779)), ('n01871265', 'tusker', np.float32(0.14579865)), ('n02504013', 'Indian_elephant', np.float32(0.03874171))]


## [2] ResNet50 모델 불러오기

In [4]:
resnet = ResNet50(weights='imagenet', include_top=True)
resnet.summary()

Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 25,636,712 (97.80 MB)

 Trainable params: 25,583,592 (97.59 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [5]:
img_res = image.load_img(img_path, target_size=(224, 224))
x_res = image.img_to_array(img_res)
x_res = np.expand_dims(x_res, axis=0)
x_res = tf.keras.applications.resnet50.preprocess_input(x_res)
preds_res = resnet.predict(x_res)
print('ResNet50 예측 결과:', tf.keras.applications.resnet50.decode_predictions(preds_res, top=3)[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
ResNet50 예측 결과: [('n02504458', 'African_elephant', np.float32(0.6553752)), ('n01871265', 'tusker', np.float32(0.24361861)), ('n02504013', 'Indian_elephant', np.float32(0.10070144))]


## [3] 모델 튜닝 (Fine-tuning)

In [6]:
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(150,150,3))
base_model.trainable = False

model_ft = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])

model_ft.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_ft.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inception_v3 (Functional)       │ (None, 3, 3, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,065,701 (84.17 MB)

 Trainable params: 262,917 (1.00 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

In [7]:
for layer in base_model.layers[-30:]:
    layer.trainable = True
model_ft.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

## [4] 전이학습 (Transfer Learning)

In [8]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

feature_model = ResNet50(weights='imagenet', include_top=False, input_shape=(32,32,3))
feature_model.trainable = False

transfer_model = models.Sequential([
    feature_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

transfer_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
transfer_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 1, 1, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,114,826 (91.99 MB)

 Trainable params: 527,114 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [9]:
history = transfer_model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
test_loss, test_acc = transfer_model.evaluate(x_test, y_test, verbose=0)
print(f'전이학습 테스트 정확도: {test_acc:.4f}')

Epoch 1/3
352/352 - 125s - 354ms/step - accuracy: 0.1336 - loss: 2.2807 - val_accuracy: 0.2264 - val_loss: 2.1841
Epoch 2/3
352/352 - 145s - 413ms/step - accuracy: 0.1655 - loss: 2.1774 - val_accuracy: 0.2494 - val_loss: 2.0794
Epoch 3/3
352/352 - 129s - 365ms/step - accuracy: 0.1754 - loss: 2.1528 - val_accuracy: 0.2588 - val_loss: 2.0359
전이학습 테스트 정확도: 0.2507


In [10]:
models_info = {
    'InceptionV3': inception.count_params(),
    'ResNet50': resnet.count_params(),
    'Fine-tuned Inception': model_ft.count_params(),
    'Transfer ResNet50': transfer_model.count_params()
}
for name, params in models_info.items():
    print(f'{name}: {params:,} parameters')

InceptionV3: 23,851,784 parameters
ResNet50: 25,636,712 parameters
Fine-tuned Inception: 22,065,701 parameters
Transfer ResNet50: 24,114,826 parameters


---
## 📝 [실습 7-3 수행 결과 정리]

> 실행 환경: TensorFlow 2.21.0 / Keras 3.15.1 (CPU 2코어). 코드는 원본 그대로 실행했습니다. 전이학습 정확도가 낮은 원인은 노트북과 **같은 ResNet50(동결) + 같은 분류 헤드**를 쓰고 입력 전처리·해상도만 바꿔 따로 검증했습니다. 추론 시간은 `tf.function`으로 컴파일한 뒤 1장씩 10회 측정한 중앙값입니다.

### 1) 사전학습 모델 3종 비교 (같은 코끼리 사진, 7-2의 VGG16 포함)

| 모델 | 입력 | Conv 층 수 | 파라미터 | 분류층(FC) 비중 | 연산량 (MAC/장) | CPU 추론 (1장) | ImageNet Top-1 (Keras 문서) | 코끼리 사진 1위 확률 |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| VGG16 | 224 | 13 | 1억 3,836만 | 89.4% | 15.5 G | 217 ms | 71.3% | **0.843** |
| InceptionV3 | 299 | 94 | 2,385만 | 8.6% | 5.7 G | **80 ms** | **77.9%** | 0.712 |
| ResNet50 | 224 | 53 | 2,564만 | 8.0% | **3.9 G** | 84 ms | 74.9% | 0.655 |

- 세 모델 모두 1~3위가 African_elephant / tusker / Indian_elephant로 같고, 확률값도 교수님 원본 출력과 같습니다.
- InceptionV3·ResNet50은 VGG16보다 Conv 층이 4~7배 많지만 **파라미터는 약 1/5~1/6, 연산량은 37%·25% 수준이고, 속도는 약 2.6배 빠르며 정확도는 더 높습니다.** FC 대신 **Global Average Pooling** 뒤에 분류층 하나만 두고(분류층 비중 8%), **1×1 Conv**로 채널을 줄인 뒤 큰 커널을 쓰며(Inception), **Skip connection**으로 깊은 망을 학습하기 때문입니다(강의 31~34쪽).
- 이 사진 한 장의 확률은 VGG16이 가장 높지만 ImageNet 전체 정확도 순위는 반대입니다. **확신도(확률)는 정확도가 아니며**, 모델 비교는 데이터셋 단위로 해야 합니다.

### 2) Fine-tuning 셀 (InceptionV3, 150×150 입력, 5클래스 헤드)

| 시점 | 학습 대상 파라미터 | 동결 파라미터 |
|:---|:---:|:---:|
| `model_ft.summary()` (전부 동결) | 262,917 (헤드만) | 21,802,784 |
| 마지막 30층 해제 + lr 1e-5 재컴파일 후 | **5,418,949** (헤드 + 해제된 층 5,156,032) | 16,646,752 |

- 마지막 30층은 `mixed10` 블록 일부(Conv 8, BatchNorm 9, Activation 9, Concatenate 3, AvgPool 1)입니다. 임의 데이터로 1 step 학습해 보니 해제된 Conv 가중치가 학습률과 같은 1e-5만큼 바뀌어, **Keras 3에서는 실제로 학습 대상이 된 것**을 확인했습니다. (summary는 해제 이전에 출력되어 262,917로 보입니다.)
- 주의할 점
  1. `base_model.trainable`은 여전히 `False`이고 `base_model.trainable_weights`는 비어 있습니다. Keras 3는 변수 단위 플래그를 보기 때문에 동작하지만, Keras 2(tf.keras 2.x)에서는 상위 모델이 `trainable=False`면 하위 층 설정과 관계없이 전부 동결되어 **같은 코드로 아무 층도 풀리지 않습니다.** 버전과 무관한 표준 패턴은 `base_model.trainable = True`로 연 다음 `base_model.layers[:-30]`을 다시 `False`로 얼리는 방식입니다.
  2. 해제된 30층에 **BatchNormalization 9개**가 들어 있습니다. 작은 배치로 미세조정하면 사전학습된 이동평균 통계가 흔들리므로, Keras 전이학습 가이드처럼 `base_model(inputs, training=False)`로 호출해 BN을 추론 모드로 두는 것이 안전합니다.
  3. 이 노트북에는 5클래스 데이터와 `fit`이 없어 **모델 구성까지만** 실습합니다. 실제로는 ① 헤드만 먼저 학습 → ② 상위층을 풀고 작은 학습률로 재학습하는 2단계로 진행합니다.

### 3) 전이학습 (ResNet50 동결 + CIFAR-10) — 정확도 0.25의 원인 분석

| Epoch | Train Acc | Val Acc | Val Loss |
|:---:|:---:|:---:|:---:|
| 1 | 0.1336 | 0.2264 | 2.1841 |
| 2 | 0.1655 | 0.2494 | 2.0794 |
| 3 | 0.1754 | 0.2588 | 2.0359 |

- **테스트 정확도 0.2507** (교수님 원본 0.2450). 무작위 추측(0.10)보다 조금 나은 수준으로, 사실상 학습에 실패한 상태입니다. 예측도 bird 2,455장·horse 1,851장에 몰리고 cat은 63장, deer는 3장뿐입니다.

**원인 ① — 입력 전처리 불일치 (가장 큰 원인)**

ResNet50의 ImageNet 가중치는 0~255 픽셀에서 채널 평균을 빼고 BGR 순서로 바꾼 입력(`resnet50.preprocess_input`)으로 학습되었습니다. 노트북은 `/255`만 해서 넣습니다. 테스트 1,000장의 특징 벡터(GAP 출력 2048차원)를 비교하면 다음과 같습니다.

| 입력 방식 | 값 범위 | 이미지 간 평균 코사인 유사도 | 한 번이라도 활성화된 특징 차원 |
|:---|:---:|:---:|:---:|
| 노트북 (`x / 255`) | 0 ~ 1 | **0.995** | 13.8% |
| `preprocess_input(x * 255)` | −124 ~ 151 | 0.233 | 100% |

- 입력 값의 범위가 학습 때(−124~151)와 전혀 다른 0~1이라서 **어떤 이미지를 넣어도 거의 같은 특징 벡터**가 나옵니다(코사인 0.995, 2048차원 중 86%는 항상 0). 분류 헤드가 구분할 정보가 없으니 학습을 오래 해도 정확도가 오르지 않습니다.

**원인 ② — 32×32 해상도**

ResNet50은 입력을 1/32로 줄이므로 32×32 입력은 마지막에 **1×1×2048**이 됩니다(ImageNet 224 입력은 7×7). 224px 사진에서 학습한 질감·부분 형태 특징이 32px 물체에서는 제대로 나오지 않습니다.

**검증 실험** — 같은 ResNet50(동결) + 같은 헤드(Dense 256 → Dropout 0.5 → Dense 10), Adam, batch 128, validation_split 0.1

| 구성 | 전처리 | 입력 (마지막 특징맵) | 학습 / 테스트 | 3 Epoch | 10 Epoch |
|:---|:---|:---:|:---:|:---:|:---:|
| 노트북 원본 실행 | `/255` | 32 (1×1) | 45,000 / 10,000 | 0.251 | – |
| ⓐ 노트북 설정 재현 | `/255` | 32 (1×1) | 45,000 / 10,000 | 0.291 | 0.319 |
| ⓑ 전처리만 수정 | `preprocess_input` | 32 (1×1) | 45,000 / 10,000 | **0.649** | 0.674 |
| ⓑ' 같은 설정, 데이터 축소 | `preprocess_input` | 32 (1×1) | 9,000 / 2,000 | 0.600 | 0.626 |
| ⓒ + 64px 업샘플 | `preprocess_input` | 64 (2×2) | 9,000 / 2,000 | 0.741 | 0.752 |
| ⓓ + 96px 업샘플 | `preprocess_input` | 96 (3×3) | 9,000 / 2,000 | **0.822** | 0.847 |
| (참고) 7-1 구조 CNN을 처음부터 학습 | `/255` | 32 | 45,000 / 10,000 | 0.585 | – |

> 업샘플 실험은 CPU 시간 때문에 학습 10,000장(이 중 9,000장 학습, 1,000장 검증)과 테스트 2,000장으로만 했습니다. ⓑ'는 같은 조건에서 해상도 효과만 비교하기 위한 기준입니다. ⓐ 0.291과 노트북 0.251의 차이는 초기값 차이입니다.

- **전처리 한 줄이 가장 큰 원인입니다.** 같은 모델·같은 헤드·같은 3 Epoch에서 0.29 → 0.65(+36%p)로 오릅니다. 원본 설정은 10 Epoch를 돌려도 0.32에 머물러, "학습 부족"이 아니라 "입력이 틀린" 문제임을 보여 줍니다.
- **해상도가 두 번째 원인입니다.** 같은 9,000장으로 32 → 64 → 96px로 키우면 0.60 → 0.74 → 0.82로 오릅니다.
- 원본 설정(0.25)은 **처음부터 학습한 작은 CNN(0.585)보다도 낮습니다.** 사전학습 모델의 이점은 그 모델의 입력 조건을 맞춰야 생기며, 맞추면 데이터 1/5(9,000장)만으로도 0.82까지 올라갑니다.

### 4) 실행하며 발견한 코드 이슈
- **전처리:** `x_train / 255.0` 대신 `tf.keras.applications.resnet50.preprocess_input(x_train * 255.0)`(또는 0~255 원본에 바로 적용)을 써야 합니다. 가능하면 `layers.Resizing(96, 96)` 등으로 업샘플해서 넣는 것이 좋습니다.
- **예제 이미지 URL:** 셀 [5]의 `https://storage.googleapis.com/download.tensorflow.org/example_images/elephant.jpg`는 현재 **403(AccessDenied)**입니다. `get_file('elephant.jpg', ...)`은 캐시 폴더(`~/.keras/datasets/`)에 같은 이름의 파일이 있으면 다운로드를 건너뛰므로, **실습 7-2를 먼저 실행해 hackaday 코끼리 사진이 캐시된 경우에만** 동작합니다. 교수님 원본과 이번 실행 모두 이 경우이며, 7-3만 새 PC에서 단독 실행하면 셀 [5]에서 오류가 납니다. 7-2와 같은 URL로 바꾸면 해결됩니다.
- **Fine-tuning:** 2)의 주의할 점 3가지(`base_model.trainable` 플래그, BN 처리, 학습 데이터·`fit` 없음).
- CPU 2코어에서 전이학습 1 Epoch 약 2~2.5분(교수님 환경 약 1분)이 걸렸습니다.

### 5) 핵심 정리
1. **Inception(1×1 Conv, 다중 스케일)과 ResNet(Skip connection)**은 VGG보다 훨씬 깊으면서 파라미터는 약 1/5~1/6입니다. FC를 GAP로 바꾼 효과가 가장 큽니다.
2. 사전학습 모델을 쓸 때는 **그 모델이 학습한 입력 형식(전처리 함수·해상도)을 그대로 맞추는 것**이 가장 중요합니다. 이번 실습에서는 전처리 한 줄로 정확도가 0.25 → 0.65, 해상도까지 맞추면 0.82가 됩니다.
3. 특징 추출(헤드만 학습)은 빠르고 적은 데이터로도 되며, 미세조정은 상위층까지 작은 학습률로 조정해 도메인 차이를 줄입니다. 불량 이미지가 적은 제조 현장에서는 **ImageNet 사전학습 모델 + 올바른 전처리 + 헤드 학습**이 현실적인 출발점입니다.